# MaintAI — Task 2 + Task 3
## Vector Database & Hybrid Retrieval Engine + LLM Prompt Engineering & Web Fallback

**Member 2 responsibility (Task 2):** dense embeddings, ChromaDB, hybrid (exact code + semantic) retrieval, device/manual filtering, ranking, deduplication, and a RAG context builder.

**Task 3 (added in this revision):** strict grounded-prompt LLM generation on top of Task 2's `hybrid_retrieve()` / `build_rag_context()`, a grounding/validation guard, and an official-source web fallback for questions the manuals don't answer.

**This notebook is restart-safe and checkpoint-based.** After a Colab crash or manual restart, re-run Phases 0–5 (Configuration through Embedding Model) to reconnect to your existing persisted data on Drive — no PDF re-processing, re-chunking, or re-embedding happens unless the underlying files are genuinely missing. Task 3 (Phase 14+) needs no API key to run at all — it defaults to a no-network stub backend so the full pipeline is testable immediately.

Reference source: built from the existing `Task_1_(7)` notebook's metadata structure and already-processed manuals (SC 6002XL, MAGNETOM Skyra, and others in the same Drive folder) — not from assumed field names.

---
## Phase 0 — Configuration

All paths and settings in one place. Nothing here does any work — it's read by every later phase, so change it here once rather than hunting through cells.

In [ ]:
import os

# --- Persistent project root locally ---
PROJECT_DIR = "."

DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHROMA_PATH = os.path.join(PROJECT_DIR, "Maintience", "Maintience NTI", "chroma_db")
OUTPUTS_DIR = os.path.join(PROJECT_DIR, "outputs")
CHECKPOINTS_DIR = os.path.join(PROJECT_DIR, "checkpoints")

# Manuals live in a folder locally.
MANUALS_DIR = os.path.join(PROJECT_DIR, "Maintience", "Maintience NTI")

# Existing processed chunks from Member 1's pipeline (or the preprocessing
# fallback cell below, used only on a genuinely fresh setup).
CHUNKS_JSON = os.path.join(DATA_DIR, "all_device_fault_chunks.json")

# Chroma collection settings
COLLECTION_NAME = "device_fault_chunks"

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# Retrieval defaults
TOP_K = 5
SEMANTIC_K = 5
MIN_SEMANTIC_SCORE = 0.3

print("Configuration set.")
print("PROJECT_DIR   :", PROJECT_DIR)
print("MANUALS_DIR   :", repr(MANUALS_DIR))
print("CHUNKS_JSON   :", CHUNKS_JSON)
print("CHROMA_PATH   :", CHROMA_PATH)
print("COLLECTION    :", COLLECTION_NAME)
print("EMBED MODEL   :", EMBEDDING_MODEL_NAME)


Configuration set.
PROJECT_DIR   : /content/drive/MyDrive/MaintAI
MANUALS_DIR   : '/content/drive/MyDrive/MaintAI/maintaince ' <- trailing space is intentional
CHUNKS_JSON   : /content/drive/MyDrive/MaintAI/data/all_device_fault_chunks.json
CHROMA_PATH   : /content/drive/MyDrive/MaintAI/chroma_db
COLLECTION    : device_fault_chunks
EMBED MODEL   : all-MiniLM-L6-v2


In [ ]:
import glob

# Non-recursive on purpose. A recursive glob previously walked into nested
# maintaince/maintaince/maintaince/... duplicate paths under MyDrive. All 44
# manuals live directly inside MANUALS_DIR (one level deep), so a flat,
# non-recursive glob is both correct and safe here.
pdf_files_preview = glob.glob(os.path.join(MANUALS_DIR, "*.pdf"))

print(f"PDF manuals found: {len(pdf_files_preview)}")
for i, pdf in enumerate(pdf_files_preview, 1):
    print(f"{i}. {pdf}")


PDF manuals found: 0


---
## Phase 1 — Environment / Dependencies

Pins Pillow **before** anything else installs it as a sub-dependency — the old notebook's crash (`ImportError: cannot import name '_Ink' from PIL._typing`) came from a mismatched/corrupted Pillow install being pulled in indirectly by `transformers`. Installing a known-good Pillow version first, with `--no-cache-dir`, avoids a stale cached wheel causing the same mismatch again.

This phase does **not** load any model — it only installs packages and verifies imports. Loading the embedding model happens in Phase 5; Qwen/LLM packages are isolated in the optional Phase 14 and are never required for Task 2 to run.

In [ ]:
# Pin Pillow first, with no cache, to avoid the PIL._typing._Ink ImportError
# seen in the previous notebook (stale/corrupted cached wheel mismatch).
!pip install --no-cache-dir "pillow>=10.4.0" --quiet

# Core Task 2 dependencies — stable, minimal set. No Qwen/bitsandbytes/tavily here.
!pip install --quiet "chromadb>=0.5.0" "sentence-transformers>=3.0.0" "transformers>=4.40.0" --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
# Health check — confirms imports work before we touch Drive or any data.
import importlib

required = ["PIL", "chromadb", "sentence_transformers", "transformers"]
for module_name in required:
    importlib.import_module(module_name)

print("Environment OK")


Environment OK


---
## Phase 2 — Google Drive

Mounts Drive once and ensures the persistent folder structure exists. All expensive/generated artifacts (chunks JSON, Chroma DB, outputs) live here so they survive a runtime restart.

In [ ]:
import os

# Google Colab drive mount is commented out for local usage.
# from google.colab import drive
# drive.mount('/content/drive')

for d in [PROJECT_DIR, DATA_DIR, CHROMA_PATH, OUTPUTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project folders ready:")
for d in [PROJECT_DIR, DATA_DIR, CHROMA_PATH, OUTPUTS_DIR, CHECKPOINTS_DIR]:
    print(" -", d)


Mounted at /content/drive
Project folders ready:
 - /content/drive/MyDrive/MaintAI
 - /content/drive/MyDrive/MaintAI/data
 - /content/drive/MyDrive/MaintAI/chroma_db
 - /content/drive/MyDrive/MaintAI/outputs
 - /content/drive/MyDrive/MaintAI/checkpoints


---
## Phase 3 — Load Existing Processed Data

**Do not re-run PDF extraction/chunking by default.** If `all_device_fault_chunks.json` already exists on Drive (from the old notebook's pipeline or Member 1's work), we load it directly. The expensive preprocessing cell below only runs if you explicitly call it — it is never part of the normal Run All path.

Chunk structure (as actually produced by the existing pipeline — verified from the old notebook, not assumed):
```
{
  "chunk_id": "...",
  "content": "...",
  "type": "text" | "table",
  "metadata": {
    "device": "...",
    "error_codes": ["E37", ...],
    "manual": "...",
    "page": 66,
    "has_safety_warning": true
  }
}
```

In [ ]:
def load_existing_chunks(chunks_json=CHUNKS_JSON):
    """Loads already-processed chunks from Drive. Returns None if not found."""
    if os.path.exists(chunks_json):
        with open(chunks_json, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

import json
chunks_data = load_existing_chunks()

if chunks_data is not None:
    print(f"Loaded existing chunks: {len(chunks_data)} chunks from {CHUNKS_JSON}")
else:
    print(f"No existing chunks found at {CHUNKS_JSON}.")
    print("Run the OPTIONAL preprocessing cell below only if this is a fresh setup.")


No existing chunks found at /content/drive/MyDrive/MaintAI/data/all_device_fault_chunks.json.
Run the OPTIONAL preprocessing cell below only if this is a fresh setup.


### Optional — preprocessing fallback (only runs if `chunks_data` is `None` above)

This is the extraction/chunking logic carried over from the old notebook (PyMuPDF text
extraction, pdfplumber table extraction, fault/error-code detection, safety-warning
detection, chunking, metadata), fixed to point at the real manuals path and to trigger
**automatically and only** when no processed chunks exist yet — never on a normal restart.

The previous version of this section additionally had a second, broken preprocessing cell
that called an undefined `extract_pdf()` function and used the wrong manuals path
(`PROJECT_DIR/"manuals"` instead of the real `MANUALS_DIR`). That cell has been removed —
this single cell is now the one and only preprocessing implementation.

In [ ]:
# Runs automatically (and only) when no processed chunks exist yet — this is
# what makes preprocessing restart-safe: once all_device_fault_chunks.json
# exists on Drive, this cell is a no-op on every later run, so PDFs are never
# re-processed unnecessarily.

RUN_PREPROCESSING = chunks_data is None

if RUN_PREPROCESSING:
    !pip install PyMuPDF --quiet
    !pip install pdfplumber --quiet
    import fitz  # PyMuPDF
    import pdfplumber
    import re

    FAULT_PATTERN = re.compile(r'\b(E\d{2,4}|ERR-\d{2,4}|Fault\s*\d{1,4}|Code\s*\d{1,4})\b', re.IGNORECASE)

    def find_all_fault_codes(text):
        matches = {m.upper() for m in FAULT_PATTERN.findall(text)}
        return sorted(matches) if matches else ["GENERAL"]

    def extract_tables_with_bboxes(plumber_page):
        results = []
        try:
            tables = plumber_page.find_tables()
            for t in tables:
                table_data = t.extract()
                if not table_data or len(table_data) < 2:
                    continue
                header = [str(cell).replace('\n', ' ') if cell else '' for cell in table_data[0]]
                md_lines = ["| " + " | ".join(header) + " |",
                            "| " + " | ".join(["---"] * len(header)) + " |"]
                for row in table_data[1:]:
                    clean_row = [str(cell).replace('\n', ' ') if cell else '' for cell in row]
                    md_lines.append("| " + " | ".join(clean_row) + " |")
                results.append({"markdown": "\n".join(md_lines), "bbox": t.bbox})
        except Exception:
            pass
        return results

    def extract_page_text_excluding_tables(fitz_page, table_bboxes, top_margin=40, bottom_margin=40):
        rect = fitz_page.rect
        body_crop = fitz.Rect(rect.x0, rect.y0 + top_margin, rect.x1, rect.y1 - bottom_margin)
        if not table_bboxes:
            return fitz_page.get_text("text", clip=body_crop)
        words = fitz_page.get_text("words", clip=body_crop)

        def in_any_table(x0, y0, x1, y1):
            for (tx0, ty0, tx1, ty1) in table_bboxes:
                if x0 >= tx0 - 2 and x1 <= tx1 + 2 and y0 >= ty0 - 2 and y1 <= ty1 + 2:
                    return True
            return False

        kept_words = [w for w in words if not in_any_table(w[0], w[1], w[2], w[3])]
        kept_words.sort(key=lambda w: (round(w[1], 1), w[0]))
        lines, current_line, current_y = [], [], None
        for w in kept_words:
            y = round(w[1], 1)
            if current_y is None or abs(y - current_y) < 2:
                current_line.append(w[4]); current_y = y
            else:
                lines.append(" ".join(current_line)); current_line = [w[4]]; current_y = y
        if current_line:
            lines.append(" ".join(current_line))
        return "\n\n".join(lines)

    def process_single_manual(pdf_path):
        doc = fitz.open(pdf_path)
        filename = os.path.basename(pdf_path)
        device_name = filename.rsplit('.', 1)[0].replace('_', ' ')
        extracted_chunks = []
        with pdfplumber.open(pdf_path) as plumber_doc:
            for page_index in range(len(doc)):
                page_num = page_index + 1
                page = doc[page_index]
                plumber_page = plumber_doc.pages[page_index]

                table_entries = extract_tables_with_bboxes(plumber_page)
                table_bboxes = [te["bbox"] for te in table_entries]
                for table_entry in table_entries:
                    table_md = table_entry["markdown"]
                    codes = find_all_fault_codes(table_md)
                    has_warning = any(w in table_md.upper() for w in ["WARNING", "DANGER", "CAUTION"])
                    extracted_chunks.append({
                        "chunk_id": f"{device_name}_P{page_num}_TBL_{len(extracted_chunks)+1}",
                        "content": table_md, "type": "table",
                        "metadata": {"device": device_name, "error_codes": codes, "manual": filename,
                                     "page": page_num, "has_safety_warning": has_warning}
                    })

                page_text = extract_page_text_excluding_tables(page, table_bboxes)
                if not page_text.strip():
                    continue
                paragraphs = re.split(r'\n\s*\n', page_text)
                for para in paragraphs:
                    clean_para = para.strip().replace('\n', ' ')
                    if len(clean_para) < 35 and not any(w in clean_para.upper() for w in ["WARNING", "DANGER", "CAUTION"]):
                        continue
                    if not clean_para:
                        continue
                    codes = find_all_fault_codes(clean_para)
                    has_warning = any(w in clean_para.upper() for w in ["WARNING", "DANGER", "CAUTION"])
                    extracted_chunks.append({
                        "chunk_id": f"{device_name}_P{page_num}_TXT_{len(extracted_chunks)+1}",
                        "content": clean_para, "type": "text",
                        "metadata": {"device": device_name, "error_codes": codes, "manual": filename,
                                     "page": page_num, "has_safety_warning": has_warning}
                    })
        doc.close()
        return extracted_chunks

    def run_batch_manuals_pipeline(manuals_directory, output_json=CHUNKS_JSON):
        pdf_files = [f for f in os.listdir(manuals_directory) if f.lower().endswith('.pdf')]
        print(f"Starting processing for {len(pdf_files)} PDF files...")
        all_chunks = []
        for idx, pdf_file in enumerate(pdf_files, 1):
            full_path = os.path.join(manuals_directory, pdf_file)
            try:
                chunks = process_single_manual(full_path)
                all_chunks.extend(chunks)
                print(f"[{idx}/{len(pdf_files)}] Processed '{pdf_file}' -> {len(chunks)} chunks.")
            except Exception as e:
                print(f"[{idx}/{len(pdf_files)}] Failed '{pdf_file}': {e}")
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(all_chunks, f, indent=4, ensure_ascii=False)
        print(f"Saved {len(all_chunks)} chunks to '{output_json}'.")
        return all_chunks

    # IMPORTANT: use MANUALS_DIR (with its trailing space) from Phase 0 config —
    # never PROJECT_DIR/"manuals", and never a recursive search.
    chunks_data = run_batch_manuals_pipeline(MANUALS_DIR)
else:
    n = len(chunks_data) if chunks_data else 0
    print(f"Preprocessing skipped — using existing chunks_data ({n} chunks).")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 123.6 MB/s eta 0:00:00
Starting processing for 44 PDF files...
[1/44] Processed 'CT_Examination_Operator_Manual_-_Somatom_Scope_-_VC50_SAPEDM_CT-Examination-Operator-Manual-Scope_EN_11517070.02.pdf' -> 1148 chunks.
[2/44] Processed 'Service Manual.pdf' -> 354 chunks.
[3/44] Processed 'Wheels Manual.pdf' -> 135 chunks.
[4/44] Processed 'Siemens Industry.pdf' -> 30 chunks.
[5/44] Processed 'Siemens Ag 2017.pdf' -> 277 chunks.
[6/44] Processed 'Ge Healthcare.pdf' -> 1166 chunks.
[7/44] Processed 'Ge Healthcare
Image Vault.pdf' -> 712 chunks.
[8/44] 

---
## Phase 4 — Build OR Load ChromaDB

Connects to the **persistent** Chroma collection on Drive. If it already contains documents, we use it as-is — no re-embedding, no rebuild. Only indexes when the collection is genuinely empty.

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

existing_count = collection.count()
print(f"Connected to collection '{COLLECTION_NAME}' at {CHROMA_PATH}")
print(f"Existing document count: {existing_count}")

NEEDS_INDEXING = existing_count == 0
if NEEDS_INDEXING:
    print("Collection is empty — indexing is needed (see next cell).")
else:
    print("Collection already populated — skipping re-indexing.")


Connected to collection 'device_fault_chunks' at /content/drive/MyDrive/MaintAI/chroma_db
Existing document count: 0
Collection is empty — indexing is needed (see next cell).


---
## Phase 5 — Load Embedding Model

Loaded once, reused by every retrieval function below — never reloaded inside a function.
**Runs before indexing** (next cell) so `embedding_model` always exists before anything
calls `.encode()` on it — this was previously out of order and caused a `NameError` the
first time indexing ran on a fresh collection.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def embed_texts(texts, batch_size=64):
    """Encodes a list of strings into embedding vectors. Single shared embedder."""
    return embedding_model.encode(
        texts, batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True
    ).tolist()

print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_2305/1470635080.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


### Indexing (only runs if the collection was empty above)

In [ ]:
def prepare_documents_for_chroma(chunks):
    """
    Converts chunk dicts into the three parallel lists collection.add()/upsert()
    expects. Chroma metadata values must be str/int/float/bool, so the list-valued
    error_codes field is joined into a comma string.
    """
    ids, documents, metadatas = [], [], []
    for chunk in chunks:
        meta = chunk["metadata"]
        error_codes = meta.get("error_codes", [])
        if isinstance(error_codes, list):
            error_codes = ",".join(error_codes)
        ids.append(chunk["chunk_id"])
        documents.append(chunk["content"])
        metadatas.append({
            "device": meta.get("device", "unknown"),
            "manual": meta.get("manual", "unknown"),
            "page": int(meta.get("page", 0)),
            "error_codes": error_codes,
            "has_safety_warning": bool(meta.get("has_safety_warning", False)),
            "type": chunk.get("type", "text"),
        })
    return ids, documents, metadatas


def populate_chroma_collection(ids, documents, metadatas, batch_size=256):
    """Embeds and upserts in batches so we don't blow memory on large chunk sets."""
    total = len(ids)
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch_docs = documents[start:end]
        batch_embeddings = embedding_model.encode(
            batch_docs, batch_size=64, show_progress_bar=False, convert_to_numpy=True
        ).tolist()
        collection.upsert(
            ids=ids[start:end], documents=batch_docs,
            metadatas=metadatas[start:end], embeddings=batch_embeddings,
        )
        print(f"Upserted {end}/{total} chunks.")


if NEEDS_INDEXING:
    if chunks_data is None:
        raise RuntimeError(
            "Collection is empty AND no chunks_data is loaded. "
            "Run the Phase 3 preprocessing cell first, or copy an existing "
            "all_device_fault_chunks.json into DATA_DIR."
        )
    # embedding_model was already loaded in Phase 5 above, so this is always
    # safe to run in a normal top-to-bottom pass.
    ids, documents, metadatas = prepare_documents_for_chroma(chunks_data)
    populate_chroma_collection(ids, documents, metadatas)
    print(f"Collection now holds {collection.count()} documents.")
else:
    print("Skipping indexing — collection already populated.")


Upserted 256/20785 chunks.
Upserted 512/20785 chunks.
Upserted 768/20785 chunks.
Upserted 1024/20785 chunks.
Upserted 1280/20785 chunks.
Upserted 1536/20785 chunks.
Upserted 1792/20785 chunks.
Upserted 2048/20785 chunks.
Upserted 2304/20785 chunks.
Upserted 2560/20785 chunks.
Upserted 2816/20785 chunks.
Upserted 3072/20785 chunks.
Upserted 3328/20785 chunks.
Upserted 3584/20785 chunks.
Upserted 3840/20785 chunks.
Upserted 4096/20785 chunks.
Upserted 4352/20785 chunks.
Upserted 4608/20785 chunks.
Upserted 4864/20785 chunks.
Upserted 5120/20785 chunks.
Upserted 5376/20785 chunks.
Upserted 5632/20785 chunks.
Upserted 5888/20785 chunks.
Upserted 6144/20785 chunks.
Upserted 6400/20785 chunks.
Upserted 6656/20785 chunks.
Upserted 6912/20785 chunks.
Upserted 7168/20785 chunks.
Upserted 7424/20785 chunks.
Upserted 7680/20785 chunks.
Upserted 7936/20785 chunks.
Upserted 8192/20785 chunks.
Upserted 8448/20785 chunks.
Upserted 8704/20785 chunks.
Upserted 8960/20785 chunks.
Upserted 9216/20785 chu

---
## Checkpoint Status

Run this any time — especially right after a restart — to see what's already in place before running further phases.

In [ ]:
def print_checkpoint_status():
    chunks_found = os.path.exists(CHUNKS_JSON)
    chroma_found = os.path.exists(CHROMA_PATH)
    try:
        doc_count = collection.count()
        collection_found = True
    except NameError:
        doc_count = "N/A"
        collection_found = False
    try:
        embed_ready = embedding_model is not None
    except NameError:
        embed_ready = False

    print("=" * 40)
    print("MaintAI CHECKPOINT STATUS")
    print("=" * 40)
    print(f"Chunks JSON     : {'FOUND' if chunks_found else 'MISSING'} ({CHUNKS_JSON})")
    print(f"Chroma DB path  : {'FOUND' if chroma_found else 'MISSING'} ({CHROMA_PATH})")
    print(f"Collection      : {'FOUND' if collection_found else 'NOT CONNECTED'}")
    print(f"Document count  : {doc_count}")
    print(f"Embedding model : {'READY' if embed_ready else 'NOT LOADED'}")
    print("=" * 40)

print_checkpoint_status()


MaintAI CHECKPOINT STATUS
Chunks JSON     : FOUND (/content/drive/MyDrive/MaintAI/data/all_device_fault_chunks.json)
Chroma DB path  : FOUND (/content/drive/MyDrive/MaintAI/chroma_db)
Collection      : FOUND
Document count  : 20785
Embedding model : READY


---
## Phase 6 — Semantic Retrieval

Embeds the query, searches Chroma, supports optional device filtering, returns structured results with `chunk_id` retained so results merge cleanly with exact search later in Phase 8.

In [ ]:
# Sentinel so we can tell "caller didn't pass max_distance" (-> apply the
# Phase 0 MIN_SEMANTIC_SCORE floor automatically) apart from "caller
# explicitly passed max_distance=None" (-> no filtering, e.g. for debugging).
_UNSET = object()


def semantic_search(query, device_name=None, k=SEMANTIC_K, max_distance=_UNSET):
    """
    Semantic retrieval over the Chroma collection.
    Returns a list of dicts: text, manual_name, page_number, section_name,
    device, retrieval_type, score, chunk_id.

    max_distance: cosine-distance ceiling (lower = stricter, higher = looser).
    Defaults to `1 - MIN_SEMANTIC_SCORE` (Phase 0) so weak/irrelevant
    nearest-neighbor matches are excluded automatically. Pass
    max_distance=None explicitly to disable filtering entirely.
    """
    if max_distance is _UNSET:
        max_distance = 1 - MIN_SEMANTIC_SCORE

    where_filter = {"device": device_name} if device_name else None

    results = collection.query(
        query_embeddings=embed_texts([query]),
        n_results=k,
        where=where_filter,
    )

    ids = results.get("ids", [[]])[0]
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]

    out = []
    for cid, doc, meta, dist in zip(ids, docs, metas, dists):
        if max_distance is not None and dist > max_distance:
            continue
        out.append({
            "chunk_id": cid,
            "text": doc,
            "manual_name": meta.get("manual", "unknown"),
            "page_number": meta.get("page", "unknown"),
            "section_name": meta.get("type", "unknown"),
            "device": meta.get("device", "unknown"),
            "retrieval_type": "semantic",
            "score": 1 - dist,  # cosine distance -> similarity-style score, higher = better
        })
    return out


---
## Phase 7 — Exact Error-Code Retrieval

Metadata structure was inspected directly (Phase 3 above) rather than assumed: `error_codes` is stored as a comma-joined string in Chroma's metadata, and the raw code also appears in the chunk's own text (that's where `find_all_fault_codes()` pulled it from during ingestion). We match against the document text via Chroma's `$contains`, which is a reliable substring match without needing to parse the comma-joined metadata field.

In [ ]:
import re


# ================================================================
# QUERY-SIDE ERROR CODE DETECTION
# ================================================================

_QUERY_CODE_PATTERN = re.compile(
    r'\b(?:ERR(?:OR)?[\s\-]*CODE|ERR(?:OR)?|CODE|FAULT|E)'
    r'[\s\-]*0*(\d{1,7})\b',
    re.IGNORECASE,
)


def extract_error_codes(query):
    """
    Detect explicit error/fault codes in the query.

    Examples:
        E37
        E 37
        error E37
        error code E37
        error code 37
        code 37
        fault 37
        error 37

    Returns normalized numeric codes.
    """

    matches = {
        m for m in _QUERY_CODE_PATTERN.findall(query)
    }

    return sorted(
        matches,
        key=lambda c: (len(c), c)
    )


# ================================================================
# NORMALIZATION
# ================================================================

def _normalize_error_code(code):
    """
    Normalize codes such as:
        037 -> 37
        0037 -> 37
    """

    return str(int(code))


# ================================================================
# TABLE ROW VALIDATION
# ================================================================

def _row_is_error_entry(row, code):
    """
    Determine whether a table row represents an actual error-code
    entry rather than a generic numbered item.

    Valid example:

        | 37 | EXP_FLOW_MTR_RANGE_ERR | N/A |

    Invalid example:

        | 37 | 2090-0352 | 10.4" TFT Color Display |
    """

    code = _normalize_error_code(code)

    if "|" not in row:
        return False

    cells = [
        c.strip()
        for c in row.split("|")
        if c.strip()
    ]

    if len(cells) < 2:
        return False

    # ------------------------------------------------------------
    # The first meaningful cell must be the requested code.
    # ------------------------------------------------------------

    if not re.fullmatch(
        rf"0*{re.escape(code)}",
        cells[0]
    ):
        return False

    rest = cells[1:]

    # ------------------------------------------------------------
    # Strong textual error indicators.
    # ------------------------------------------------------------

    error_terms = (
        "error",
        "err",
        "fault",
        "alarm",
        "failure",
        "fail",
        "timeout",
        "range_err",
        "overheat",
        "overload",
        "exceeded",
    )

    for cell in rest:

        low = cell.lower()

        if any(
            term in low
            for term in error_terms
        ):
            return True

    # ------------------------------------------------------------
    # Technical identifiers.
    #
    # Examples:
    #   EXP_FLOW_MTR_RANGE_ERR
    #   POWER_COMM_ERR
    #   SENSOR_FAILURE
    # ------------------------------------------------------------

    for cell in rest:

        if not re.fullmatch(
            r"[A-Z0-9]+(?:_[A-Z0-9]+)+",
            cell
        ):
            continue

        upper = cell.upper()

        if any(
            marker in upper
            for marker in (
                "_ERR",
                "_ERROR",
                "_FAULT",
                "_FAIL",
                "_FAILURE",
                "_ALARM",
                "_TIMEOUT",
                "_EXCEEDED",
            )
        ):
            return True

    return False


# ================================================================
# EXACT MATCH REGEX
# ================================================================

def _build_exact_match_regex(code):
    """
    Build a validator for explicit error-code formats.

    Examples:
        E37
        E 37
        ERR37
        ERR-37
        ERROR 37
        FAULT 37
        CODE 37
    """

    c = re.escape(
        _normalize_error_code(code)
    )

    pattern = (
        rf"\bE\s?0*{c}\b"
        rf"|\bERR(?:OR)?[\s\-]*0*{c}\b"
        rf"|\bFAULT[\s\-]*0*{c}\b"
        rf"|\bERROR\s+CODE[\s\-]*0*{c}\b"
        rf"|\bCODE[\s\-]*0*{c}\b"
    )

    return re.compile(
        pattern,
        re.IGNORECASE
    )


# ================================================================
# EXTRACT THE ACTUAL MATCHING ROW
# ================================================================

def _extract_code_evidence(text, code):
    """
    Extract the specific row/line containing the requested code.

    The complete chunk is still retained separately as context.
    """

    code = _normalize_error_code(code)

    # ------------------------------------------------------------
    # First: line-based matching
    # ------------------------------------------------------------

    for line in text.splitlines():

        stripped = line.strip()

        if not stripped:
            continue

        # Markdown table row
        if "|" in stripped:

            if _row_is_error_entry(
                stripped,
                code
            ):
                return stripped

        # Explicit formats such as:
        # E37 ...
        # ERROR 37 ...
        # FAULT 37 ...
        explicit = re.search(
            rf"\b(?:E|ERR(?:OR)?|FAULT|ERROR\s+CODE|CODE)"
            rf"[\s\-]*0*{re.escape(code)}\b.*",
            stripped,
            re.IGNORECASE
        )

        if explicit:
            return stripped

    # ------------------------------------------------------------
    # Fallback for flattened table text.
    # ------------------------------------------------------------

    table_pattern = re.compile(
        rf"\|\s*0*{re.escape(code)}\s*\|.*?(?=\||\n|\Z)",
        re.IGNORECASE
    )

    match = table_pattern.search(text)

    if match:
        candidate = match.group(0)

        if _row_is_error_entry(
            candidate,
            code
        ):
            return candidate.strip()

    return None


# ================================================================
# EXACT ERROR SEARCH
# ================================================================

def exact_error_search(
    query_or_code,
    device_name=None,
    k=TOP_K
):
    """
    Exact error-code retrieval.

    Finds the correct chunk, validates that the requested code is
    actually an error-code entry, and extracts the matching row.

    Generic:
        E37
        E42
        E100
        E999999
        etc.

    No code is hardcoded.
    """

    codes = extract_error_codes(
        query_or_code
    )

    if not codes:
        return []

    where_filter = (
        {"device": device_name}
        if device_name
        else None
    )

    matched = {}

    for code in codes:

        code = _normalize_error_code(
            code
        )

        validator = _build_exact_match_regex(
            code
        )

        # --------------------------------------------------------
        # Candidate retrieval.
        #
        # Chroma first finds chunks containing the numeric code.
        # The validator below performs the real validation.
        # --------------------------------------------------------

        try:

            candidates = collection.get(
                where=where_filter,
                where_document={
                    "$contains": code
                },
                include=[
                    "documents",
                    "metadatas"
                ],
            )

        except Exception:
            continue

        ids = candidates.get(
            "ids",
            []
        ) or []

        docs = candidates.get(
            "documents",
            []
        ) or []

        metas = candidates.get(
            "metadatas",
            []
        ) or []

        # --------------------------------------------------------
        # Validate candidates.
        # --------------------------------------------------------

        for cid, doc, meta in zip(
            ids,
            docs,
            metas
        ):

            if not doc:
                continue

            if cid in matched:
                continue

            # ----------------------------------------------------
            # Explicit error-code formats.
            # ----------------------------------------------------

            explicit_match = validator.search(
                doc
            )

            # ----------------------------------------------------
            # Table-row format.
            # Example:
            # | 37 | EXP_FLOW_MTR_RANGE_ERR | N/A |
            # ----------------------------------------------------

            table_match = False

            for line in doc.splitlines():

                if _row_is_error_entry(
                    line,
                    code
                ):
                    table_match = True
                    break

            if not explicit_match and not table_match:
                continue

            # ----------------------------------------------------
            # Extract the actual evidence row.
            # ----------------------------------------------------

            evidence = _extract_code_evidence(
                doc,
                code
            )

            if not evidence:
                continue

            matched[cid] = {
                "chunk_id": cid,

                # Full chunk = context for RAG
                "text": doc,

                # Exact row = actual evidence
                "matched_text": evidence,

                "manual_name": meta.get(
                    "manual",
                    "unknown"
                ),

                "page_number": meta.get(
                    "page",
                    "unknown"
                ),

                "section_name": meta.get(
                    "type",
                    "unknown"
                ),

                "device": meta.get(
                    "device",
                    "unknown"
                ),

                "retrieval_type": "exact",

                "score": 1.0,

                "matched_code": code,
            }

    return list(
        matched.values()
    )[:k]

---
## Phase 8 — Hybrid Retrieval

```
USER QUERY
    |
Detect error codes
    |
    +------------------+------------------+
    |                                     |
Exact Error Search                 Semantic Search
    |                                     |
    +------------------+------------------+
                       |
                Merge Results
                       |
                  Deduplicate
                       |
                     Rank
                       |
                    Top-K
```
Device filtering applies to **both** retrieval paths. Exact matches always outrank semantic matches for the same chunk.

In [ ]:
def hybrid_retrieve(
    query,
    device_name=None,
    top_k=TOP_K,
    min_semantic_score=MIN_SEMANTIC_SCORE
):
    """
    Hybrid retrieval.

    - Explicit error codes -> exact retrieval is authoritative.
    - Symptom/context -> semantic retrieval.
    - Exact results always rank first.
    - Duplicate chunks are merged by chunk_id.
    - A nonexistent explicit error code must not produce unrelated
      semantic results.
    """

    # ---------------------------------------------------------
    # 1. Detect explicit error codes
    # ---------------------------------------------------------
    error_codes = extract_error_codes(query)
    has_error_code = bool(error_codes)

    # ---------------------------------------------------------
    # 2. Exact retrieval
    # ---------------------------------------------------------
    exact_results = exact_error_search(
        query,
        device_name=device_name,
        k=top_k
    )

    # ---------------------------------------------------------
    # 3. Build semantic query
    #
    # Remove explicit error-code expressions so semantic search
    # focuses on the actual symptom/context.
    # ---------------------------------------------------------
    semantic_query = query

    if has_error_code:
        semantic_query = re.sub(
            r'\b(?:ERR(?:OR)?[\s\-]*CODE|ERR(?:OR)?|CODE|FAULT|E)'
            r'[\s\-]*0*\d{1,7}\b',
            ' ',
            query,
            flags=re.IGNORECASE
        )

        semantic_query = re.sub(
            r'\s+',
            ' ',
            semantic_query
        ).strip()

    # ---------------------------------------------------------
    # 4. Semantic retrieval
    # ---------------------------------------------------------
    semantic_results = []

    # If this is ONLY an explicit error-code query and there is
    # no exact evidence, do NOT retrieve unrelated semantic docs.
    if not has_error_code or exact_results or semantic_query:
        # For an error-code-only query with no exact result,
        # semantic_query will normally be empty, so this won't run.
        if semantic_query:
            semantic_max_distance = (
                None
                if min_semantic_score is None
                else 1 - min_semantic_score
            )

            semantic_results = semantic_search(
                semantic_query,
                device_name=device_name,
                k=top_k,
                max_distance=semantic_max_distance
            )
        elif not has_error_code:
            semantic_max_distance = (
                None
                if min_semantic_score is None
                else 1 - min_semantic_score
            )

            semantic_results = semantic_search(
                query,
                device_name=device_name,
                k=top_k,
                max_distance=semantic_max_distance
            )

    # ---------------------------------------------------------
    # 5. If explicit error code exists but exact search failed
    # and there is no additional symptom/context -> NOT FOUND
    # ---------------------------------------------------------
    if has_error_code and not exact_results and not semantic_query:
        return []

    # ---------------------------------------------------------
    # 6. Merge + deduplicate
    # ---------------------------------------------------------
    combined = {}

    # Semantic first
    for result in semantic_results:
        combined[result["chunk_id"]] = result

    # Exact overwrites semantic on collision
    for result in exact_results:
        combined[result["chunk_id"]] = result

    # ---------------------------------------------------------
    # 7. Exact first, then semantic score
    # ---------------------------------------------------------
    ranked = sorted(
        combined.values(),
        key=lambda r: (
            r["retrieval_type"] != "exact",
            -r["score"]
        )
    )

    return ranked[:top_k]


---
## Phase 9 — Device / Manual Filtering

Device names come directly from the metadata already stored in Chroma (derived dynamically from each PDF's filename during ingestion — never hardcoded to a single device). This helper lists what's actually indexed, so `device_name=` values are always real, not guessed.

In [ ]:
import re

def list_available_devices():
    """Returns the distinct device names actually present in the collection."""
    sample = collection.get(limit=min(collection.count(), 5000))
    devices = sorted({m.get("device", "unknown") for m in sample.get("metadatas", [])})
    return devices


_HASH_LIKE = re.compile(r'^[0-9A-Fa-f]{16,}$')


def pick_demo_device(devices):
    """
    Some source PDFs are named with an anonymized hash instead of a real
    device name (e.g. "C56F1635D2314C7E9E90B4A40163B5C6"), and a hash sorts
    first alphabetically -- so naively using devices[0] for a demo/test can
    silently filter to an irrelevant manual. This skips hash-like names and
    returns the first human-readable device name instead (falls back to
    devices[0] if every entry looks hash-like).
    """
    for d in devices:
        if not _HASH_LIKE.match(d.strip()):
            return d
    return devices[0] if devices else None


available_devices = list_available_devices()
print(f"Devices currently indexed ({len(available_devices)}):")
for d in available_devices:
    print(" -", d)

demo_device = pick_demo_device(available_devices)
print(f"\nDefault demo device for filtered tests: {demo_device!r}")


Devices currently indexed (10):
 - C56F1635D2314C7E9E90B4A40163B5C6
 - CT Examination Operator Manual - Somatom Scope - VC50 SAPEDM CT-Examination-Operator-Manual-Scope EN 11517070.02
 - Ge Healthcare
 - Ge Healthcare
Image Vault
 - Mribookv1.3
 - Philips V24, V25 Agilent (M1205) Monitor   Service Manual
 - Service Manual
 - Siemens Ag 2017
 - Siemens Industry
 - Wheels Manual

Default demo device for filtered tests: 'CT Examination Operator Manual - Somatom Scope - VC50 SAPEDM CT-Examination-Operator-Manual-Scope EN 11517070.02'


---
## Phase 10 — Ranking & Deduplication

Already implemented inside `hybrid_retrieve()` (Phase 8) — documented here separately per the required notebook structure. Priority order:

1. Exact error-code match
2. Device/manual match (enforced via the `where` filter on both search paths, not a ranking step — non-matching-device results never enter the candidate pool)
3. Semantic relevance (cosine similarity score)
4. Deduplication by `chunk_id` — a chunk returned by both paths keeps its exact-search identity, never counted twice

No ML reranker — kept simple and explainable, as specified.

---
## Phase 11 — RAG Context Builder

The interface between Task 2 and the downstream LLM. Output field names match what Member 3's prompt templates expect (`text`, `manual_name`, `page_number`, `section_name`, `device`). Never invents content — only formats what was retrieved.

In [ ]:
def build_rag_context(results):
    """
    Builds grounded, citable context from hybrid_retrieve() results.
    Returns "NOT_FOUND_IN_MANUAL" if results is empty.
    """
    if not results:
        return "NOT_FOUND_IN_MANUAL"

    blocks = []
    for i, r in enumerate(results, start=1):
        blocks.append(
            f"[Source {i}]\n"
            f"Device: {r.get('device', 'unknown')}\n"
            f"Manual: {r.get('manual_name', 'unknown')}\n"
            f"Page: {r.get('page_number', 'unknown')}\n"
            f"Section: {r.get('section_name', 'unknown')}\n"
            f"Retrieval: {r.get('retrieval_type', 'unknown')} (score={r.get('score', 0):.3f})\n\n"
            f"Text:\n{r.get('text', '')}\n"
        )
    return "\n---\n\n".join(blocks)


---
## Phase 12 — Retrieval Testing

In [ ]:
def print_results(label, results):
    print(f"\n=== {label} ===")

    if not results:
        print("No results -- NOT_FOUND_IN_MANUAL")
        return

    for r in results:
        print(
            f"[{r['retrieval_type']}] "
            f"score={r['score']:.3f} | "
            f"device={r['device']} | "
            f"manual={r['manual_name']} | "
            f"page={r['page_number']} | "
            f"section={r['section_name']}"
        )

        # For exact retrieval, show the actual matching error-code row.
        if r["retrieval_type"] == "exact" and r.get("matched_text"):
            print(f"    match: {r['matched_text']}")
        else:
            preview = r["text"][:150].replace("\n", " ")
            print(f"    text: {preview}...")


# TEST 1 -- explicit error code
print_results(
    "TEST 1: What does E37 mean?",
    hybrid_retrieve("What does E37 mean?")
)


# TEST 2 -- symptom only, no code
print_results(
    "TEST 2: The monitor is overheating",
    hybrid_retrieve("The monitor is overheating.")
)


# TEST 3 -- symptom only, different topic
print_results(
    "TEST 3: NBP measurement not working",
    hybrid_retrieve("The NBP measurement is not working.")
)


# TEST 4 -- code + symptom combined
print_results(
    "TEST 4: E37 + overheating",
    hybrid_retrieve(
        "E37 appears and the monitor is overheating."
    )
)


# TEST 5 -- same query, device filtered
print_results(
    "TEST 5: Same query, device filtered",
    hybrid_retrieve(
        "E37 appears and the monitor is overheating.",
        device_name=demo_device
    )
)


# TEST 6 -- nonexistent error code
print_results(
    "TEST 6: Nonexistent code E999999",
    hybrid_retrieve("E999999")
)


=== TEST 1: What does E37 mean? ===
[exact] score=1.000 | device=Service Manual | manual=Service Manual.pdf | page=53 | section=table
    match: | 37 | EXP_FLOW_MTR_RANGE_ERR | N/A |
[semantic] score=0.442 | device=CT Examination Operator Manual - Somatom Scope - VC50 SAPEDM CT-Examination-Operator-Manual-Scope EN 11517070.02 | manual=CT_Examination_Operator_Manual_-_Somatom_Scope_-_VC50_SAPEDM_CT-Examination-Operator-Manual-Scope_EN_11517070.02.pdf | page=205 | section=text
    text: means storage on a hard disk. Be aware of the fact that any...
[semantic] score=0.398 | device=philips Advanced Visualization Workspace Version 16.0.3 2026-08-11 | manual=philips Advanced Visualization Workspace Version 16.0.3 2026-08-11.pdf | page=84 | section=text
    text: means it is physically installed or connected to your system, like the hard drive, a CD drive, or a...
[semantic] score=0.394 | device=Av Cardiovascular | manual=Av Cardiovascular.pdf | page=19 | section=text
    text: The following

---
## Phase 13 — Retrieval Evaluation

In [ ]:
# A small representative eval set: query -> a keyword expected in a genuinely
# relevant retrieved chunk. Simple, explainable proxy for correctness -- not
# a substitute for manual review. Numbers below are only ever measured live,
# never fabricated.
EVAL_QUERIES = [
    {"query": "What does E37 mean?", "expected_keyword": "E37"},
    {"query": "The monitor is overheating.", "expected_keyword": "temperature"},
    {"query": "The NBP measurement is not working.", "expected_keyword": "NBP"},
]

def evaluate_retrieval(eval_queries, k=TOP_K):
    print(f"{'Query':45} {'Top-1':6} {'Top-3':6} {'Top-5':6}")
    for item in eval_queries:
        results = hybrid_retrieve(item["query"], top_k=k)
        keyword = item["expected_keyword"].upper()
        hit_ranks = [idx for idx, r in enumerate(results) if keyword in r["text"].upper()]
        top1 = "YES" if 0 in hit_ranks else "no"
        top3 = "YES" if any(r < 3 for r in hit_ranks) else "no"
        top5 = "YES" if hit_ranks else "no"
        print(f"{item['query'][:43]:45} {top1:6} {top3:6} {top5:6}")

evaluate_retrieval(EVAL_QUERIES)


Query                                         Top-1  Top-3  Top-5 
What does E37 mean?                           no     no     no    
The monitor is overheating.                   no     no     no    
The NBP measurement is not working.           YES    YES    YES   


---
## Phase 14 — Task 3: LLM Prompt Engineering & Grounded Generation

Task 3 turns Task 2's retrieval output into a technician-facing answer. It **never
re-retrieves anything itself** — it consumes `hybrid_retrieve()` and `build_rag_context()`
exactly as Task 2 produces them.

```
User Query
    |
hybrid_retrieve()        <- Task 2 (unchanged)
    |
build_rag_context()      <- Task 2 (unchanged)
    |
Prompt Template           <- Task 3
    |
LLM  (modular backend: stub / anthropic / openai / local Qwen)
    |
Grounding / Validation Guard   <- Task 3 (Phase 15)
    |
NOT_FOUND_IN_MANUAL? --> Web Fallback   <- Task 3 (Phase 16)
    |
Final structured answer   <- Task 3 (Phase 17)
```

**The LLM backend is modular and isolated from Task 2.** Nothing here touches `collection`,
`embedding_model`, `hybrid_retrieve`, or `build_rag_context` — it only calls those functions
as black boxes, exactly as they're defined above. No API keys are hard-coded anywhere;
everything comes from environment variables, and with the default `LLM_PROVIDER = "stub"`
the entire pipeline is runnable and testable with zero API keys and no network access.

In [ ]:
import os

# ------------------------------------------------------------------
# Modular LLM backend selection. Change this one line (or set the env var
# before running this cell) to swap models/providers later without
# touching any other Task 3 code.
#
#   "stub"       -> no API key needed, no network call. Deterministic,
#                   purely extractive: never invents text. Lets the whole
#                   pipeline (retrieval -> validation -> web fallback) be
#                   tested with zero credentials.
#   "anthropic"  -> Claude via the Anthropic API (ANTHROPIC_API_KEY env var)
#   "openai"     -> GPT via the OpenAI API (OPENAI_API_KEY env var)
#   "qwen_local" -> local 4-bit Qwen2.5-7B-Instruct, carried over from the
#                   team's earlier Task 3 notebook and kept fully isolated
#                   below — it does not install, download, or load
#                   anything unless this is explicitly selected.
# ------------------------------------------------------------------
LLM_PROVIDER = os.environ.get("MAINTAI_LLM_PROVIDER", "stub")

# API keys — NEVER hard-code these here. Set them in the Colab environment
# before running this cell, e.g.:
#   import os; os.environ["ANTHROPIC_API_KEY"] = "..."
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "")  # used by Phase 16 web fallback

ANTHROPIC_MODEL = os.environ.get("MAINTAI_ANTHROPIC_MODEL", "claude-sonnet-4-5")
OPENAI_MODEL = os.environ.get("MAINTAI_OPENAI_MODEL", "gpt-4o-mini")

print(f"LLM_PROVIDER = '{LLM_PROVIDER}'")
if LLM_PROVIDER == "anthropic" and not ANTHROPIC_API_KEY:
    print("WARNING: LLM_PROVIDER='anthropic' but ANTHROPIC_API_KEY is not set.")
if LLM_PROVIDER == "openai" and not OPENAI_API_KEY:
    print("WARNING: LLM_PROVIDER='openai' but OPENAI_API_KEY is not set.")
if LLM_PROVIDER == "qwen_local":
    print("NOTE: 'qwen_local' loads a 7B model — needs a Colab GPU runtime.")


LLM_PROVIDER = 'stub'


In [ ]:
# Lightweight Task 3 dependencies only — kept in a separate install cell so
# Task 2 (Phase 1) never depends on Task 3 packages, and vice versa.
!pip install --quiet anthropic openai requests --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.9 MB/s eta 0:00:00


### Prompt templates

In [ ]:
MANUAL_SYSTEM_PROMPT = """You are MaintAI, a technical maintenance assistant for field technicians.

The RETRIEVED MANUAL CONTEXT below is your ONLY source of truth. Follow these rules exactly:

1. Answer using ONLY information present in the retrieved manual context. Never invent
   troubleshooting steps, never infer a procedure that is not written there, never guess
   what an error code means.
2. If the context does not contain enough information to answer the query, respond with
   EXACTLY the single line: NOT_FOUND_IN_MANUAL
   Do not add anything else in that case.
3. Safety comes first. If the context contains any DANGER, WARNING, or CAUTION notice
   relevant to this query, show it at the very top of your answer, in that priority order
   (DANGER before WARNING before CAUTION). Never omit a safety notice that applies to the
   requested procedure.
4. Always cite where the information came from: Device, Manual, Page, and Section, using
   the [Source N] labels already present in the context.
5. Keep the answer concise and technician-friendly — short steps, no filler, no repeating
   the entire context back verbatim.
6. Give concise step-by-step troubleshooting instructions only when the manual context
   actually supports them.

Retrieved Manual Context:
{rag_context}

Technician Query:
{query}

Answer:"""

WEB_FALLBACK_LABEL = "WEB FALLBACK — NOT FOUND IN MANUAL"

WEB_FALLBACK_SYSTEM_PROMPT = """You are MaintAI. The technician's question could NOT be
answered from the uploaded service manuals. You are given web search results instead,
which should already be biased toward official manufacturer sources.

Rules:
1. Start your answer with the exact label: """ + WEB_FALLBACK_LABEL + """
2. Base your answer ONLY on the provided search results below — do not use outside
   knowledge.
3. Clearly prioritize official manufacturer / manufacturer-support sources over any other
   source in the results.
4. If the search results do not give a clear, trustworthy answer, respond with exactly:
   """ + WEB_FALLBACK_LABEL + """
   No information could be verified from an official source.
5. Include the source title/URL for any claim you make.
6. If this concerns a medical device, do NOT present unofficial web results as if they were
   manufacturer instructions — clearly flag that this is unverified against the manual.
7. Keep the answer concise and technician-friendly.

Web Search Results:
{search_results}

Technician Query:
{query}

Answer:"""

print("Prompt templates ready.")


Prompt templates ready.


### Modular LLM dispatcher

`generate_grounded_answer(query, rag_context)` is the single Task 3 entry point that
consumes Task 2's `build_rag_context()` output directly — it never retrieves anything
itself. Every backend is reached through one dispatch function, `call_llm()`, so swapping
providers never requires touching the prompt logic above or `process_query()` below.

In [ ]:
def _call_anthropic(prompt, max_tokens=800):
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    resp = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text").strip()


def _call_openai(prompt, max_tokens=800):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content.strip()


def _call_stub(prompt, max_tokens=800):
    """
    No-API deterministic backend used for testing the pipeline end-to-end
    without any credentials. Purely extractive — it never invents text: it
    returns NOT_FOUND_IN_MANUAL when the prompt has no retrieved sources in
    it, or a clearly-labeled placeholder otherwise, so the surrounding
    plumbing (grounding validator, web-fallback trigger, etc.) can be
    exercised safely without a real model.
    """
    if "[Source" not in prompt:
        return "NOT_FOUND_IN_MANUAL"
    return ("[stub backend — no LLM configured] Grounded answer would be generated here "
            "from the retrieved manual context above. Set LLM_PROVIDER to 'anthropic', "
            "'openai', or 'qwen_local' for a real generated answer.")


def call_llm(prompt, max_tokens=800):
    """Single dispatch point — every Task 3 function calls this, never a provider SDK directly."""
    if LLM_PROVIDER == "anthropic":
        return _call_anthropic(prompt, max_tokens)
    if LLM_PROVIDER == "openai":
        return _call_openai(prompt, max_tokens)
    if LLM_PROVIDER == "qwen_local":
        return _call_qwen_local(prompt, max_tokens)  # defined in the isolated cell below
    return _call_stub(prompt, max_tokens)


def generate_grounded_answer(query, rag_context):
    """
    Task 3 entry point. Takes Task 2's rag_context (the exact string returned
    by build_rag_context()) — never retrieves anything itself. Returns the
    raw LLM answer text, which may be exactly "NOT_FOUND_IN_MANUAL".
    """
    if rag_context == "NOT_FOUND_IN_MANUAL":
        return "NOT_FOUND_IN_MANUAL"
    prompt = MANUAL_SYSTEM_PROMPT.format(rag_context=rag_context, query=query)
    return call_llm(prompt)

print("generate_grounded_answer() ready. Active backend:", LLM_PROVIDER)


generate_grounded_answer() ready. Active backend: stub


### Optional — local Qwen2.5-7B-Instruct backend (isolated)

Carried over from the team's earlier Task 3 notebook. Kept fully isolated: this cell only
installs packages, downloads weights, and touches GPU memory if `LLM_PROVIDER == "qwen_local"`.
It is never a side effect of running Task 2 or the rest of Task 3.

In [ ]:
if LLM_PROVIDER == "qwen_local":
    !pip install --quiet transformers accelerate bitsandbytes

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    QWEN_MODEL_NAME = os.environ.get("MAINTAI_QWEN_MODEL", "Qwen/Qwen2.5-7B-Instruct")

    _bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
    _qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME, quantization_config=_bnb_config, device_map="auto",
    )

    def _call_qwen_local(prompt, max_tokens=800):
        messages = [{"role": "user", "content": prompt}]
        input_ids = _qwen_tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(_qwen_model.device)
        output_ids = _qwen_model.generate(
            input_ids, max_new_tokens=max_tokens, temperature=0.1,
            do_sample=False, pad_token_id=_qwen_tokenizer.eos_token_id,
        )
        return _qwen_tokenizer.decode(
            output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True
        ).strip()

    print(f"Loaded local backend: {QWEN_MODEL_NAME}")
else:
    def _call_qwen_local(prompt, max_tokens=800):
        raise RuntimeError(
            "LLM_PROVIDER is not 'qwen_local' — set LLM_PROVIDER='qwen_local' in the "
            "Phase 14 config cell and re-run this cell to enable the local backend."
        )
    print("Skipped — LLM_PROVIDER != 'qwen_local' (local Qwen not loaded).")


Skipped — LLM_PROVIDER != 'qwen_local' (local Qwen not loaded).


---
## Phase 15 — Grounding / Validation

A guard that runs on every LLM answer before it reaches the technician. It never trusts the
LLM's self-reported status — it checks the answer text against what Task 2 actually
retrieved, catching fabricated error codes, missing safety warnings, dropped citations, and
a mismatch between "nothing was retrieved" and "the answer claims to know something".

In [ ]:
import re

FAULT_CODE_PATTERN = re.compile(r'\b(E\d{2,5}|ERR-\d{2,5}|Fault\s*\d{1,5}|Code\s*\d{1,5})\b', re.IGNORECASE)
SAFETY_WORDS = ("DANGER", "WARNING", "CAUTION")


def validate_grounding(answer, rag_context, results):
    """
    Checks an LLM answer against the retrieved context it was supposed to be
    grounded in. Returns {"status": ..., "issues": [...]}.
    status is one of: "FOUND_IN_MANUAL", "NOT_FOUND_IN_MANUAL", "FLAGGED".
    """
    issues = []

    if rag_context == "NOT_FOUND_IN_MANUAL":
        if answer.strip() != "NOT_FOUND_IN_MANUAL":
            issues.append("Retrieval found nothing, but the answer does not say NOT_FOUND_IN_MANUAL.")
            return {"status": "FLAGGED", "issues": issues}
        return {"status": "NOT_FOUND_IN_MANUAL", "issues": issues}

    if answer.strip() == "NOT_FOUND_IN_MANUAL":
        return {"status": "NOT_FOUND_IN_MANUAL", "issues": issues}

    # From here on, results is non-empty and rag_context has real content.
    context_upper = rag_context.upper()
    answer_upper = answer.upper()

    # 1. Fabricated error codes: codes in the answer that never appear in the retrieved context.
    answer_codes = {m.upper() for m in FAULT_CODE_PATTERN.findall(answer)}
    context_codes = {m.upper() for m in FAULT_CODE_PATTERN.findall(rag_context)}
    fabricated = answer_codes - context_codes
    if fabricated:
        issues.append(f"Answer references error code(s) not present in retrieved context: {sorted(fabricated)}")

    # 2. Missing safety warnings: if context has DANGER/WARNING/CAUTION, the answer should too.
    for word in SAFETY_WORDS:
        if word in context_upper and word not in answer_upper:
            issues.append(f"Context contains a '{word}' notice missing from the answer.")

    # 3. Citation check — answer should reference a source marker, manual name, or the stub label.
    has_citation = (
        "[SOURCE" in answer_upper
        or "STUB BACKEND" in answer_upper
        or any(str(r.get("manual_name", "")).upper() in answer_upper for r in results)
    )
    if not has_citation:
        issues.append("Answer does not appear to cite a manual/source from the retrieved context.")

    status = "FLAGGED" if issues else "FOUND_IN_MANUAL"
    return {"status": status, "issues": issues}

print("validate_grounding() ready.")


validate_grounding() ready.


---
## Phase 16 — Web Fallback

Triggered **only** when Task 2 retrieval genuinely came up empty (`build_rag_context()`
returned `NOT_FOUND_IN_MANUAL`) or the LLM itself decided the context wasn't enough. It
never runs for a query the manuals already answer. It prefers the manufacturer's own site,
and every web-derived answer is explicitly labeled `WEB FALLBACK — NOT FOUND IN MANUAL` so
it is never silently mixed with manual-sourced content.

In [ ]:
import requests


def web_search_official(query, device_name=None, manufacturer_domain=None, max_results=5):
    """
    Searches the web via the Tavily API, preferring an official manufacturer domain
    when one is given. Returns (formatted_results_str, source_list). Degrades
    gracefully — never raises — if TAVILY_API_KEY is not configured, so the
    pipeline stays runnable without it.
    """
    if not TAVILY_API_KEY:
        return ("No web search available — TAVILY_API_KEY is not set.", [])

    payload = {
        "api_key": TAVILY_API_KEY,
        "query": f"{device_name + ' ' if device_name else ''}{query} official manual support".strip(),
        "max_results": max_results,
        "search_depth": "advanced",
    }
    if manufacturer_domain:
        payload["include_domains"] = [manufacturer_domain]

    try:
        resp = requests.post("https://api.tavily.com/search", json=payload, timeout=20)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return (f"Web search failed: {e}", [])

    hits = data.get("results", [])
    if not hits:
        return ("No relevant official web results found.", [])

    formatted, sources = [], []
    for h in hits:
        title, url, content = h.get("title", ""), h.get("url", ""), h.get("content", "")
        formatted.append(f"Title: {title}\nURL: {url}\nContent: {content}")
        sources.append({"title": title, "url": url})
    return ("\n\n".join(formatted), sources)


def generate_web_fallback_answer(query, device_name=None, manufacturer_domain=None):
    """Runs the web search and a labeled LLM summary of it. Only ever called
    for a query the manuals could not answer."""
    search_results, sources = web_search_official(query, device_name, manufacturer_domain)
    if not sources:
        return f"{WEB_FALLBACK_LABEL}\n{search_results}", []
    prompt = WEB_FALLBACK_SYSTEM_PROMPT.format(search_results=search_results, query=query)
    answer = call_llm(prompt)
    if not answer.upper().startswith("WEB FALLBACK"):
        answer = f"{WEB_FALLBACK_LABEL}\n{answer}"
    return answer, sources

print("Web fallback ready. TAVILY_API_KEY set:", bool(TAVILY_API_KEY))


Web fallback ready. TAVILY_API_KEY set: False


---
## Phase 17 — End-to-End RAG Pipeline

Wires Task 2 and Task 3 together exactly as specified:

```
User Query -> hybrid_retrieve() -> build_rag_context() -> Prompt Template -> LLM
    -> Validation / Grounding Guard -> (web fallback only if genuinely needed) -> Final Answer
```

Retrieval happens **exactly once**, through `hybrid_retrieve()` — Task 3 never
retrieves independently, and web search is never called unless the manuals truly had
nothing relevant.

In [ ]:
def process_query(query, device_name=None, manufacturer_domain=None, top_k=TOP_K):
    """
    The complete Task 2 + Task 3 pipeline for one technician query.
    """
    results = hybrid_retrieve(query, device_name=device_name, top_k=top_k)
    rag_context = build_rag_context(results)

    sources = [
        {"manual": r["manual_name"], "page": r["page_number"], "device": r["device"],
         "retrieval_type": r["retrieval_type"]}
        for r in results
    ]

    llm_answer = generate_grounded_answer(query, rag_context)
    validation = validate_grounding(llm_answer, rag_context, results)

    used_web_fallback = False
    needs_fallback = (
        rag_context == "NOT_FOUND_IN_MANUAL"
        or llm_answer.strip() == "NOT_FOUND_IN_MANUAL"
    )

    if needs_fallback:
        used_web_fallback = True
        llm_answer, web_sources = generate_web_fallback_answer(query, device_name, manufacturer_domain)
        sources = web_sources
        status = "NOT_FOUND_IN_MANUAL"
        validation = {"status": "NOT_FOUND_IN_MANUAL", "issues": []}
    else:
        status = validation["status"]

    return {
        "answer": llm_answer,
        "status": status,
        "sources": sources,
        "used_web_fallback": used_web_fallback,
        "validation": validation,
        "rag_context": rag_context,
        "retrieved_results": results,
    }

print("process_query() ready — the full Task 2 -> Task 3 pipeline.")


process_query() ready — the full Task 2 -> Task 3 pipeline.


---
## Phase 18 — End-to-End Evaluation / Demo Tests

Runs the required test queries through the full `process_query()` pipeline. With the
default `LLM_PROVIDER = "stub"` these exercise every branch of the pipeline (found in
manual, not found → web fallback, validation flags) without needing any API key. Switch
`LLM_PROVIDER` in Phase 14 to `"anthropic"`, `"openai"`, or `"qwen_local"` for real
generated answers — no other code changes are needed anywhere in this section.

In [ ]:
def print_pipeline_result(label, result):
    print(f"\n{'='*70}\n{label}\n{'='*70}")
    print("RETRIEVAL STATUS   :", "FOUND" if result["retrieved_results"] else "NOT_FOUND_IN_MANUAL")
    print("RETRIEVED SOURCES  :")
    if result["retrieved_results"]:
        for r in result["retrieved_results"]:
            print(f"   - [{r['retrieval_type']}] {r['manual_name']} p.{r['page_number']} ({r['device']})")
    else:
        print("   (none)")
    print("RAG CONTEXT (first 300 chars):")
    print("  ", result["rag_context"][:300].replace("\n", " "))
    print("LLM ANSWER         :")
    print("  ", result["answer"][:500])
    print("WEB FALLBACK USED  :", result["used_web_fallback"])
    print("VALIDATION STATUS  :", result["validation"]["status"], "-", result["validation"]["issues"])


TASK3_TEST_QUERIES = [
    ("TEST 1 -- explicit error code", "What does E37 mean?", None),
    ("TEST 2 -- symptom only", "The monitor is overheating.", None),
    ("TEST 3 -- another maintenance issue", "The NBP measurement is not working.", None),
    ("TEST 4 -- error code + symptom", "E37 appears and the monitor is overheating.", None),
    ("TEST 5 -- nonexistent error code", "What does E999999 mean?", None),
    ("TEST 6 -- should trigger web fallback",
     "What is the latest official firmware update procedure published on the manufacturer's website?",
     None),
]

for label, query, device in TASK3_TEST_QUERIES:
    result = process_query(query, device_name=device)
    print_pipeline_result(label, result)



TEST 1 -- explicit error code
RETRIEVAL STATUS   : FOUND
RETRIEVED SOURCES  :
   - [exact] Service Manual.pdf p.53 (Service Manual)
   - [semantic] CT_Examination_Operator_Manual_-_Somatom_Scope_-_VC50_SAPEDM_CT-Examination-Operator-Manual-Scope_EN_11517070.02.pdf p.205 (CT Examination Operator Manual - Somatom Scope - VC50 SAPEDM CT-Examination-Operator-Manual-Scope EN 11517070.02)
   - [semantic] philips Advanced Visualization Workspace Version 16.0.3 2026-08-11.pdf p.84 (philips Advanced Visualization Workspace Version 16.0.3 2026-08-11)
   - [semantic] Av Cardiovascular.pdf p.19 (Av Cardiovascular)
   - [semantic] CT_Examination_Operator_Manual_-_Somatom_Scope_-_VC50_SAPEDM_CT-Examination-Operator-Manual-Scope_EN_11517070.02.pdf p.107 (CT Examination Operator Manual - Somatom Scope - VC50 SAPEDM CT-Examination-Operator-Manual-Scope EN 11517070.02)
RAG CONTEXT (first 300 chars):
   [Source 1] Device: Service Manual Manual: Service Manual.pdf Page: 53 Section: table Retrieval: exact

---
## Summary

### What changed in this revision

**Task 2 fixes (Part 1):**
- Removed the broken duplicate preprocessing cell that called an undefined `extract_pdf()`
  and pointed at the wrong manuals path — there is now exactly one preprocessing
  implementation.
- `MANUALS_DIR` is defined once, in Phase 0, as
  `"/content/drive/MyDrive/MaintAI/maintaince "` (trailing space preserved), and every
  other cell that touches the manuals folder reads that single variable instead of
  redefining or guessing the path.
- The manuals preview and the preprocessing fallback both use a **non-recursive** glob, so
  they can no longer produce nested `maintaince/maintaince/...` paths.
- Preprocessing now runs **automatically and only** when `chunks_data is None` — no PDF is
  ever re-processed on a normal restart.
- **Indexing order fixed:** the embedding model (Phase 5) now loads *before* the indexing
  cell, eliminating the previous `NameError` risk from calling `embedding_model.encode()`
  before the model existed. Indexing itself is still fully restart-safe (skips whenever the
  Chroma collection already has documents; never duplicates or re-embeds).
- All existing Task 2 interfaces are unchanged: `hybrid_retrieve(...)`,
  `build_rag_context(...)`, `collection`, `embedding_model`, `CHUNKS_JSON`, `CHROMA_PATH`.

**Task 3 added (Part 2), Phases 14–18:**
- Strict grounded-prompt templates (`MANUAL_SYSTEM_PROMPT`, `WEB_FALLBACK_SYSTEM_PROMPT`)
  enforcing manual-only answers, DANGER → WARNING → CAUTION priority, mandatory
  Device/Manual/Page/Section citations, and the `NOT_FOUND_IN_MANUAL` sentinel.
- A modular `call_llm()` dispatcher (`LLM_PROVIDER = "stub" | "anthropic" | "openai" |
  "qwen_local"`) so the model/API can change later by editing one line. No API keys are
  hard-coded — everything is read from environment variables. The default `"stub"` backend
  needs no credentials at all, so the pipeline is fully testable immediately. The team's
  earlier local-Qwen implementation was reused and kept fully isolated: it only loads if
  `LLM_PROVIDER == "qwen_local"`.
- `validate_grounding()` — a grounding guard that checks the LLM's answer against what was
  actually retrieved (fabricated error codes, missing safety warnings, missing citations,
  a "found" claim when retrieval returned nothing).
- `web_search_official()` / `generate_web_fallback_answer()` — an official-source-preferring
  web fallback (Tavily), triggered only when the manuals genuinely had nothing, and always
  labeled `WEB FALLBACK — NOT FOUND IN MANUAL`, never mixed silently with manual content.
- `process_query()` — the single orchestration function wiring
  `hybrid_retrieve() -> build_rag_context() -> generate_grounded_answer() ->
  validate_grounding() -> (web fallback if needed)` into one structured result:
  `{"answer", "status", "sources", "used_web_fallback", "validation", ...}`.
- Phase 18 runs the six required end-to-end test queries through `process_query()` and
  prints QUERY / RETRIEVAL STATUS / RETRIEVED SOURCES / RAG CONTEXT / LLM ANSWER / WEB
  FALLBACK USED / VALIDATION STATUS for each.

### Exact execution order (fresh setup)

Phase 0 → 1 → 2 → 3 (skip the optional preprocessing cell unless `chunks_data` is `None`)
→ 4 → 5 → indexing cell → checkpoint → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 (config, deps,
prompts, dispatcher — skip the isolated Qwen cell unless `LLM_PROVIDER == "qwen_local"`) →
15 → 16 → 17 → 18.

### After a runtime restart, re-run

Phases 0, 1, 2, 3 (data-loading cell only — **not** the optional preprocessing cell), 4, 5,
the indexing cell (it self-skips once the collection is populated), then 6 onward, including
Phase 14's config/prompt/dispatcher cells (cheap — no model download unless
`LLM_PROVIDER == "qwen_local"`).

### Should NOT be rerun unless the underlying data is missing

- The Phase 3 optional preprocessing cell — only if `all_device_fault_chunks.json` is
  genuinely missing from Drive.
- The indexing cell in Phase 4/5 — only if the Chroma collection is genuinely empty (it
  checks this itself via `NEEDS_INDEXING` and no-ops otherwise).
- The isolated Qwen-loading cell in Phase 14 — only if you've explicitly set
  `LLM_PROVIDER = "qwen_local"`.

### How Task 2 connects to Task 3 and the RAG pipeline

Task 2 ends at `build_rag_context()` (Phase 11) — a plain string, either a formatted,
citable block of retrieved manual excerpts or the literal sentinel `NOT_FOUND_IN_MANUAL`.
Task 3's `generate_grounded_answer(query, rag_context)` (Phase 14) takes that string as its
only input and never calls `hybrid_retrieve()` or touches Chroma itself — retrieval is
Task 2's job, generation is Task 3's job, and `process_query()` (Phase 17) is the one place
that calls both in order. If Task 2's retrieval genuinely finds nothing (or the LLM itself
says `NOT_FOUND_IN_MANUAL`), `process_query()` routes to the Phase 16 web fallback instead of
letting the LLM guess — so a technician asking about something outside the manuals gets a
clearly labeled web answer instead of a silently fabricated one.